# TalentCLEF TaskA 2025

# Carga de datos

In [1]:
import pandas as pd
import numpy as np

In [113]:
training_data = pd.read_csv("./data/TaskA/training/english/taskA_training_en.tsv", sep='\t', names=['family_id', 'id', 'title1', 'title2'])

validation_data = dict()

validation_data['corpus_elements'] = pd.read_csv("./data/TaskA/validation/english/corpus_elements", sep='\t').set_index('c_id').reset_index(drop=True)

validation_data['queries'] = pd.read_csv("./data/TaskA/validation/english/queries", sep='\t').set_index('q_id').reset_index(drop=True)

validation_data['qrels'] = pd.read_csv("./data/TaskA/validation/english/qrels.tsv", sep='\t', names=['q_id', 'iter', 'c_id', 'relevance'])[['q_id', 'c_id', 'relevance']]
validation_data['qrels']['q_id'] = validation_data['qrels']['q_id'] - 1
validation_data['qrels']['c_id'] = validation_data['qrels']['c_id'] - 1

In [114]:
validation_data['qrels']

,q_id,c_id,relevance
0,0,142,1
1,0,149,1
2,0,763,1
3,0,869,1
4,0,1463,1
...,...,...,...
2415,104,2122,1
2416,104,2143,1
2417,104,2355,1
2418,104,2399,1


In [99]:
print("Training data samples:")
print(training_data[['title1', 'title2']].head())
print("\n")

print("Validation data samples:")
print(validation_data['corpus_elements'].head())
print(validation_data['queries'].head())
print(validation_data['qrels'].head())

Training data samples:
                        title1                       title2
0                air commodore            flight lieutenant
1  command and control officer               flight officer
2                air commodore  command and control officer
3                pilot officer              squadron leader
4       royal airforce officer  command and control officer


Validation data samples:
                              jobtitle
c_id                                  
0                   recording engineer
1                 director of taxation
2     technical support representative
3                           hr manager
4              computer graphic artist
                 jobtitle
q_id                     
0                   nanny
1       food technologist
2      broadcast engineer
3     automation engineer
4            veterinarian
   q_id  iter  c_id  relevance
0     0     0   142          1
1     0     0   149          1
2     0     0   763          1
3     0    

# Aproximación con embedding ya preentrenado

Se establece device='cpu' debido a que mi ordenador no soporta CUDA

## Configuración

In [31]:
from sentence_transformers import SentenceTransformer, util

# Version multilingue del modelo
model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2', device='cpu')

In [39]:
def similarity_between_titles(title1, title2):
    emb1 = model.encode(title1, convert_to_tensor=True, device='cpu')
    emb2 = model.encode(title2, convert_to_tensor=True, device='cpu')

    similarity = util.cos_sim(emb1, emb2)

    return similarity.item()

similarity_between_titles("data scientist", "científico de datos")

0.963962197303772

In [100]:
print('Total de casos a revisar:', validation_data['corpus_elements'].shape[0] * validation_data['queries'].shape[0])

Total de casos a revisar: 274995


## Cálculo de similitud

In [91]:
# calcular la matriz de similitud de query x corpus elements
query_embeddings = model.encode(validation_data['queries']['jobtitle'].tolist()[:10], convert_to_tensor=True, device='cpu', show_progress_bar=True)
corpus_embeddings = model.encode(validation_data['corpus_elements']['jobtitle'].tolist()[:10], convert_to_tensor=True, device='cpu', show_progress_bar=True)

cosine_scores = util.cos_sim(query_embeddings, corpus_embeddings)
print('Matriz de similitud (query x corpus elements):', cosine_scores.shape)
print(cosine_scores)


Batches: 100%|██████████| 1/1 [00:00<00:00, 20.47it/s]

Matriz de similitud (query x corpus elements): torch.Size([10, 10])
tensor([[0.2130, 0.2310, 0.3083, 0.2842, 0.2359, 0.2011, 0.1473, 0.1074, 0.1654,
         0.2723],
        [0.2692, 0.1749, 0.3028, 0.4057, 0.3170, 0.2004, 0.2721, 0.1228, 0.2078,
         0.3458],
        [0.7296, 0.0904, 0.3928, 0.3502, 0.3911, 0.5527, 0.2080, 0.1380, 0.1709,
         0.1397],
        [0.4465, 0.0759, 0.4235, 0.3731, 0.4514, 0.3492, 0.2682, 0.1180, 0.1125,
         0.2792],
        [0.2493, 0.1323, 0.1978, 0.3490, 0.1692, 0.2220, 0.1964, 0.1151, 0.2831,
         0.1465],
        [0.3103, 0.4136, 0.3439, 0.4783, 0.1676, 0.4150, 0.5523, 0.2245, 0.2294,
         0.4571],
        [0.2451, 0.0838, 0.2901, 0.2779, 0.3075, 0.1779, 0.1962, 0.0208, 0.2285,
         0.1594],
        [0.2541, 0.2041, 0.0946, 0.3430, 0.1492, 0.2199, 0.2779, 0.1144, 0.1556,
         0.0089],
        [0.1760, 0.3707, 0.3571, 0.3000, 0.1952, 0.3348, 0.7368, 0.5134, 0.1664,
         0.1989],
        [0.5717, 0.1576, 0.3906, 0.3169, 

# Generacion de datos sinteticos para testear la busqueda del threshold

In [116]:
import random

# Lista para almacenar las filas
data = []

# Para cada q_id de 0 a 9
for q_id in range(10):
    # Generar entre 2 y 4 c_id aleatorios
    num_elements = random.randint(1,3)
    c_ids = random.sample(range(10), num_elements)
    
    # Crear una fila por cada c_id
    for c_id in c_ids:
        data.append({
            'q_id': q_id,
            'c_id': c_id,
            'relevance': 1
        })

# Crear el DataFrame
df = pd.DataFrame(data).sort_values(['q_id', 'c_id']).reset_index(drop=True)

print(df)
print(f"\nTotal de filas: {len(df)}")

    q_id  c_id  relevance
0      0     3          1
1      1     3          1
2      2     7          1
3      3     1          1
4      3     4          1
5      4     5          1
6      5     1          1
7      6     1          1
8      6     6          1
9      7     5          1
10     7     6          1
11     8     6          1
12     8     7          1
13     9     8          1

Total de filas: 14


In [117]:
all_combinations = []
query_size = 10
corpus_size = 10

for q_id in range(query_size):
    for c_id in range(corpus_size):
        all_combinations.append({'q_id': q_id, 'c_id': c_id})

df_all = pd.DataFrame(all_combinations)

# Hacer merge para identificar las combinaciones que faltan
df_merged = df_all.merge(df, on=['q_id', 'c_id'], how='left')
df_merged = df_merged.fillna(0).astype(int)
df_merged = df_merged.sort_values(['q_id', 'c_id']).reset_index(drop=True)

print(df_merged)

    q_id  c_id  relevance
0      0     0          0
1      0     1          0
2      0     2          0
3      0     3          1
4      0     4          0
..   ...   ...        ...
95     9     5          0
96     9     6          0
97     9     7          0
98     9     8          1
99     9     9          0

[100 rows x 3 columns]
